# Lesson 1 : Create a basic agent

In this exercise, we learn how to build and run agent in Agent Framework.

Agent Framework supports a variety of backend clients - such as, OpenAI Assistants API, OpenAI Responses API, Anthropic Client, Ollama, Azure AI agents, etc. (For other clients, you can easily build your own custom agent from scratch.)  
Throughout this workshop, we use a client to connect Microsoft Foundry (based on ```azure-ai-projects``` version 2 SDK) for building agents.

> Note : In ```azure-ai-projects``` v2 SDK, Azure OpenAI Responses API or Conversations are transparently called internally.

Firstly, we create a client object to connect Azure AI as follows.  
All the difference depending on individual clients is handled on this client's object.

In [1]:
from agent_framework.azure import AzureAIClient
from azure.identity.aio import AzureCliCredential

credential = AzureCliCredential()
client = AzureAIClient(credential=credential)

Next, create an agent using above client.

In [2]:
# define local tools
from typing import Annotated
from pydantic import Field
from random import randint

def get_weather(
    location: Annotated[str, Field(description="天気を取得する場所")],  # "the location to get the weather for"
) -> str:
    """与えられた場所の天気を取得する。"""
    conditions = ["晴れ", "曇り", "雨", "嵐"]  # "sunny", "cloudy", "rainy", "stormy"
    return f"{location}の天気は{conditions[randint(0, 3)]}です。"  # "the weather in ... is ... ."

def get_temperature(
    location: Annotated[str, Field(description="気温を取得する場所")],  # "the location to get the temperature for"
) -> str:
    """与えられた場所の気温を取得する。"""
    return f"{location}の気温は{randint(10, 30)}°Cです。"  # "the temperature in ... is ... degrees."

In [3]:
# create agent
agent = client.create_agent(
    name="BasicWeatherAgent",
    instructions="あなたは、気象情報に関する Agent です。",  # "you are an agent about weather information."
    tools=[get_weather, get_temperature]
)

Let's run the agent.  
You'll see that the function tools are being called appropriately.

> Note : By running this call, the agent is registered in Microsoft Foundry. (Because we're using ```azure-ai-projects``` client.)

In [4]:
from IPython.display import Markdown, display

result = await agent.run("大阪の天気と気温を教えてください。")  # "tell me the weather and temperature in Osaka."
display(Markdown(result.text))

大阪の天気は**雨**です。  
大阪の気温は**27°C**です。

Let's see the list of internal messages.  
You'll see that ```get_weather()``` and ```get_temperature()``` are concurrently called.

In [5]:
import agent_framework

for i, msg in enumerate(result.messages):
    print(f"********** message {i} **********")
    for c in msg.contents:
        if isinstance(c, agent_framework._types.FunctionCallContent):
            print("*** FunctionCallContent ***")
            print(f"call id : {c.call_id}")
            print(f"function name : {c.name}")
            print(f"function arguments : {c.arguments}")
        elif isinstance(c, agent_framework._types.FunctionResultContent):
            print("*** FunctionResultContent ***")
            print(f"call id : {c.call_id}")
            print(f"function result : {c.result}")
            print(f"exceptions : {c.exception}")
        elif isinstance(c, agent_framework._types.TextContent):
            print("*** TextContent ***")
            print(f"text : {c.text}")
        else:
            print(f"*** Other types : {type(c)}***")

********** message 0 **********
*** FunctionCallContent ***
call id : call_bYJEoK2MyqDFWhaFDCRL7VXv
function name : get_weather
function arguments : {"location":"大阪"}
*** FunctionCallContent ***
call id : call_hHc504M6OQJq9cAlT0Q61zMo
function name : get_temperature
function arguments : {"location":"大阪"}
********** message 1 **********
*** FunctionResultContent ***
call id : call_bYJEoK2MyqDFWhaFDCRL7VXv
function result : 大阪の天気は嵐です。
exceptions : None
*** FunctionResultContent ***
call id : call_hHc504M6OQJq9cAlT0Q61zMo
function result : 大阪の気温は19°Cです。
exceptions : None
********** message 2 **********
*** TextContent ***
text : 大阪の天気は**嵐**です。  
大阪の気温は**19°C**です。


For the fluent user experience, you can perform streaming outputs as follows.

In [6]:
async def streaming_example():
    async for chunk in agent.run_stream("大阪の天気と気温を教えてください。"):  # "tell me the weather and temperature in Osaka."
        if chunk.text:
            print(chunk.text, end="", flush=True)
    print("\n")

await streaming_example()

大阪の天気は雨です。  
大阪の気温は27°Cです。

